# Notebook 03 — Kiểm thử và xác nhận toàn luồng K-Medoids

**Cập nhật theo xác nhận của người dùng:** `cp=5`, `slope=4`, `ca=4` là mã hợp lệ. Trong phiên bản code hiện tại, `EXPECTED_VALUES` chưa bao gồm chúng; do đó bước làm sạch đang xem chúng là bất thường và điền khuyết. Bài test kiểm tra sự nhất quán này sẽ FAIL đến khi miền hợp lệ trong code được cập nhật; đó là lỗi cấu hình/pipeline, không phải kết luận dữ liệu sai.

notebook kiểm tra:

- Thuật toán K-Medoids: phép tính mẫu, chọn medoid, gán cụm, predict, seed và các trường hợp dữ liệu bất thường.
- Tiền xử lý và đánh giá: tách target, tạo đặc trưng Manhattan/Gower và tính các metric.
- Mô hình thư viện: kiểm tra adapter với dữ liệu mẫu.
- Tính nhất quán: đối chiếu dữ liệu xử lý lại với các artifact đang lưu.
- Tích hợp: thử chạy custom trên toàn bộ dữ liệu trong bộ nhớ, nhưng không lưu đè kết quả thí nghiệm.

In [ ]:
import sys
from pathlib import Path
from hashlib import sha256
import json
import numpy as np
import pandas as pd
from IPython.display import display
from tempfile import TemporaryDirectory

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "src" / "kmedoids.py").is_file()), None)
PROJECT_ROOT_FALLBACK = Path(r"C:\Users\havuh\OneDrive\Desktop\HocTap\Clustering_ of patients_with heart_disease_KMedoids")
if PROJECT_ROOT is None and (PROJECT_ROOT_FALLBACK / "src" / "kmedoids.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT_FALLBACK
if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm thấy project root có src/kmedoids.py.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.kmedoids import KMedoids
from src.data import (
    FEATURE_COLS, NUMERIC_COLS, CATEGORICAL_COLS, BINARY_COLS, TARGET_COL,
    EXPECTED_VALUES, load_heart_data, harmonize_to_uci_encoding,
    drop_duplicate_rows, replace_abnormal_with_nan, impute_mode,
    impute_median, split_features_target, data_quality_report,
)
from src.preprocessing import preprocess_features, gower_distance_matrix, save_processed
from src.library_model import fit_kmedoids
from src.evaluation import (
    silhouette, select_k, cluster_sizes, target_crosstab,
    posthoc_target_check, cluster_profile, compare_labelings, run_and_save_metrics,
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "heart_disease_whole.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
TEST_RESULTS = []
PROTECTED_PATHS = [
    RAW_PATH,
    PROCESSED_DIR / "heart_dedup.csv",
    PROCESSED_DIR / "X_manhattan.csv",
    PROCESSED_DIR / "target.csv",
    PROCESSED_DIR / "gower_distance.npy",
    METRICS_DIR / "k_selection.csv",
    METRICS_DIR / "labels.csv",
    METRICS_DIR / "medoids.csv",
    METRICS_DIR / "cross_method_ari.csv",
    METRICS_DIR / "run_config.json",
]
FILE_HASHES_BEFORE = {p: sha256(p.read_bytes()).hexdigest() for p in PROTECTED_PATHS if p.is_file()}

print("Project root:", PROJECT_ROOT)
print("Dữ liệu/artifact kiểm thử chỉ đọc; không lưu hoặc ghi đè kết quả.")

## 1. Harness kiểm thử
Mỗi test chạy riêng; lỗi một ca không chặn các ca sau. Cuối notebook tổng hợp required fail và advisory/data-contract fail.

In [ ]:
def test_case(name, fn, required=True, note=""):
    try:
        detail = fn()
        TEST_RESULTS.append({"test":name,"status":"PASS","required":required,"detail":str(detail),"note":note})
        print(f"PASS | {name} | {detail}")
        return True
    except Exception as exc:
        TEST_RESULTS.append({"test":name,"status":"FAIL","required":required,"detail":f"{type(exc).__name__}: {exc}","note":note})
        print(f"FAIL | {name} | {type(exc).__name__}: {exc}")
        return False

def expect_raises(exception_type, fn):
    try:
        fn()
    except exception_type as exc:
        return f"{type(exc).__name__}: {exc}"
    except Exception as exc:
        expected = getattr(exception_type,"__name__",str(exception_type))
        raise AssertionError(f"Expected {expected}; got {type(exc).__name__}: {exc}") from exc
    expected = getattr(exception_type,"__name__",str(exception_type))
    raise AssertionError(f"Expected {expected}; no exception was raised")

## 2. Unit tests cho custom K-Medoids
Bao gồm ví dụ 30 điểm tính tay, medoid thuộc dữ liệu, seed, `fit_predict`/`predict`, input không hợp lệ, max_iter, record trùng và các trường hợp hòa.

In [ ]:
# Các unit test chạy bằng assert, không cần pytest.
X30 = [[v] for v in list(range(1, 16)) + list(range(100, 115))]

def test_30_record_hand_example():
    m = KMedoids(2, metric="manhattan", initial_medoids=[0, 15]).fit(X30)
    assert len(X30) == 30
    assert m.medoid_indices_ == [7, 22]
    assert m.medoids_ == [[8.0], [107.0]]
    assert m.labels_ == [0] * 15 + [1] * 15
    assert m.cluster_sizes_ == [15, 15]
    assert np.isclose(m.inertia_, 112.0)
    assert m.n_iter_ == 2 and m.converged_
    return f"medoids={m.medoid_indices_}; objective={m.inertia_}; iterations={m.n_iter_}"

def test_medoids_are_actual_input_records():
    X = [[0, 0], [0, 1], [9, 9], [10, 9]]
    m = KMedoids(2, metric="manhattan", random_state=42).fit(X)
    observed = {tuple(map(float, row)) for row in X}
    assert all(tuple(row) in observed for row in m.medoids_)
    assert all(0 <= idx < len(X) for idx in m.medoid_indices_)
    return f"medoid rows={m.medoid_indices_} are members of X"

def test_seed_reproducibility():
    X = [[i, (i * 7) % 11] for i in range(20)]
    a = KMedoids(3, metric="manhattan", random_state=42).fit(X)
    b = KMedoids(3, metric="manhattan", random_state=42).fit(X)
    assert a.medoid_indices_ == b.medoid_indices_
    assert a.labels_ == b.labels_
    assert np.isclose(a.inertia_, b.inertia_)
    return f"same seed -> medoids={a.medoid_indices_}, labels/objective reproducible"

def test_euclidean_and_manhattan_metrics():
    X=[[0],[1],[9],[10]]
    expected=[0,0,1,1]
    for metric in ["euclidean","manhattan"]:
        m=KMedoids(2,metric=metric,initial_medoids=[0,2]).fit(X)
        assert m.labels_==expected and m.medoid_indices_==[0,2]
        assert np.isclose(m.inertia_,2.0) and m.converged_
    return "Euclidean and Manhattan produce expected labels, medoids and objective"

def test_fit_predict_predict_and_new_rows():
    X = [[0], [1], [2], [10], [11], [12]]
    fit = KMedoids(2, metric="manhattan", random_state=42).fit(X)
    fit_predict = KMedoids(2, metric="manhattan", random_state=42).fit_predict(X)
    assert fit_predict == fit.labels_
    assert fit.predict(X) == fit.labels_
    assert fit.predict([[0]]) == [fit.labels_[0]]
    assert fit.predict([[12]]) == [fit.labels_[-1]]
    return f"fit_predict matches labels_; predict(X_train/new) passed"

def test_predict_errors():
    expect_raises(RuntimeError, lambda: KMedoids(2).predict([[1, 2]]))
    fitted = KMedoids(2, random_state=0).fit([[0, 0], [1, 1], [9, 9]])
    expect_raises(ValueError, lambda: fitted.predict([[1]]))
    return "predict before fit and wrong feature count rejected"

def test_constructor_parameter_validation():
    expect_raises(TypeError, lambda: KMedoids(True))
    expect_raises(TypeError, lambda: KMedoids(2.0))
    expect_raises(ValueError, lambda: KMedoids(0))
    expect_raises(ValueError, lambda: KMedoids(2, max_iter=0))
    expect_raises(ValueError, lambda: KMedoids(2, metric="cosine"))
    return "invalid k type/value, max_iter, and metric rejected"

def test_input_validation_and_invalid_k():
    for bad in [[], [[], []], [[1], [2, 3]], [[np.nan], [1]], [[np.inf], [1]], [["x"], [1]]]:
        expect_raises((TypeError, ValueError), lambda bad=bad: KMedoids(1).fit(bad))
    expect_raises(ValueError, lambda: KMedoids(3).fit([[1], [2]]))
    expect_raises(ValueError, lambda: KMedoids(2, initial_medoids=[0, 0]).fit([[1], [2]]))
    expect_raises(ValueError, lambda: KMedoids(2, initial_medoids=[0, 3]).fit([[1], [2]]))
    expect_raises(ValueError, lambda: KMedoids(2, initial_medoids=[0]).fit([[1], [2]]))
    expect_raises(TypeError, lambda: KMedoids(2, initial_medoids=[0, 1.0]).fit([[1], [2]]))
    return "empty/ragged/non-numeric/non-finite data, bad k and medoid indices rejected"

def test_max_iter_limit():
    m = KMedoids(2, metric="manhattan", max_iter=1, initial_medoids=[0, 15]).fit(X30)
    assert m.n_iter_ == 1 and not m.converged_
    assert m.medoid_indices_ == [7, 22]
    assert m.labels_ == [0] * 15 + [1] * 15
    return f"max_iter=1 -> n_iter={m.n_iter_}, converged={m.converged_}; final labels/objective recomputed"

def test_duplicate_rows_and_assignment_tie():
    dup = KMedoids(2, metric="manhattan", initial_medoids=[0, 2]).fit([[0], [0], [10], [10]])
    assert dup.labels_ == [0, 0, 1, 1] and dup.medoid_indices_ == [0, 2] and dup.inertia_ == 0
    tie = KMedoids(2, metric="manhattan", initial_medoids=[0, 1]).fit([[0], [2], [1]])
    assert tie.labels_ == [0, 1, 0]  # equidistant sample -> lower cluster id
    assert tie.medoid_indices_ == [0, 1]
    return "duplicate points preserved; assignment tie resolves to lower cluster id"

def test_all_missing_imputation_fallbacks():
    frame=pd.DataFrame({"cp":[np.nan,np.nan],"age":[np.nan,np.nan]})
    filled,modes=impute_mode(frame,cols=["cp"])
    filled,medians=impute_median(filled,cols=["age"])
    assert filled["cp"].tolist()==[0,0] and modes["cp"]==0
    assert filled["age"].tolist()==[0.0,0.0] and medians["age"]==0.0
    return "all-NaN categorical/numeric columns use documented zero fallback"

def test_medoid_update_tie_policy():
    m = KMedoids(2, metric="manhattan", initial_medoids=[0, 2]).fit([[0], [2], [10]])
    assert m.medoid_indices_ == [0, 2]  # tied cost keeps old medoid when it is among minima
    assert m.converged_
    return "equal medoid costs keep previous medoid when tied"

CUSTOM_TESTS = [
    ("30-record hand calculation", test_30_record_hand_example),
    ("medoid must be observed record", test_medoids_are_actual_input_records),
    ("random seed reproducibility", test_seed_reproducibility),
    ("Euclidean and Manhattan distances", test_euclidean_and_manhattan_metrics),
    ("fit_predict and predict", test_fit_predict_predict_and_new_rows),
    ("predict input errors", test_predict_errors),
    ("constructor validation", test_constructor_parameter_validation),
    ("fit data/k/initial medoid validation", test_input_validation_and_invalid_k),
    ("max_iter and final recomputation", test_max_iter_limit),
    ("duplicate rows and assignment tie", test_duplicate_rows_and_assignment_tie),
    ("all-missing imputation fallback", test_all_missing_imputation_fallbacks),
    ("medoid-update tie policy", test_medoid_update_tie_policy),
]

for test_name, test_fn in CUSTOM_TESTS:
    test_case("CUSTOM / " + test_name, test_fn)


## 3. Loader, harmonize và xử lý dữ liệu
Kiểm tra load/filter, source remapping, khử trùng, mã bất thường, mode/median, tách target. Người dùng xác nhận `cp=5`, `slope=4`, `ca=4` là hợp lệ. Kiểm tra bên dưới phát hiện nếu `EXPECTED_VALUES` trong code chưa chứa các mã đó; khi còn thiếu, pipeline hiện coi chúng là bất thường và điền khuyết. FAIL ở đây chỉ code-contract sai lệch với xác nhận dữ liệu, không nói dữ liệu sai.

In [ ]:
def test_loader_reads_all_sources_and_filters_source():
    raw = pd.read_csv(RAW_PATH, na_values=["", " ", "?", "NA", "N/A"])
    assert len(raw) == 2283
    all_rows = load_heart_data(RAW_PATH)
    assert len(all_rows) == 2283 and "source" not in all_rows.columns
    assert list(all_rows.columns) == FEATURE_COLS + [TARGET_COL]
    source = raw["source"].iloc[0]
    filtered = load_heart_data(RAW_PATH, sources=[source])
    assert len(filtered) == int((raw["source"] == source).sum())
    return f"all sources={raw.source.nunique()}, rows={len(all_rows)}; filter {source}={len(filtered)}"

def test_loader_errors():
    expect_raises(FileNotFoundError, lambda: load_heart_data(PROJECT_ROOT / "__missing_test_file.csv"))
    expect_raises(ValueError, lambda: load_heart_data(RAW_PATH, sources=["__unknown_source__"]))
    return "missing file and unknown source rejected"

def test_harmonization_is_copy_and_source_aware():
    frame = pd.DataFrame({"cp":[1,1], "slope":[1,1], "thal":[0,0], "source":["kaggle_heart_1025","uci_multi_cleveland"]})
    out = harmonize_to_uci_encoding(frame)
    assert out is not frame
    assert frame.loc[0,"cp"] == 1 and frame.loc[1,"cp"] == 1
    assert out.loc[0,"cp"] == 2 and out.loc[0,"slope"] == 2 and out.loc[0,"thal"] == 3
    assert out.loc[1,"cp"] == 1 and out.loc[1,"slope"] == 1 and out.loc[1,"thal"] == 0
    return "Kaggle row remapped; UCI row retained; input frame not mutated"

def test_duplicate_abnormal_imputation_and_split():
    fixture = pd.DataFrame({
        "age":[40.,40.,50.], "trestbps":[120.,120.,np.nan], "chol":[200.,200.,220.],
        "thalach":[150.,150.,160.], "oldpeak":[0.,0.,1.], "cp":[1,1,2],
        "restecg":[0,0,1], "slope":[1,1,2], "ca":[0,0,1], "thal":[3,3,6],
        "sex":[0,0,1], "fbs":[0,0,1], "exang":[0,0,1], "target":[0,0,1]})
    dedup = drop_duplicate_rows(fixture)
    assert len(dedup) == 2 and dedup.index.tolist() == [0,1]
    bad = dedup.copy(); bad.loc[1,"cp"] = 9
    cleaned = replace_abnormal_with_nan(bad)
    assert pd.isna(cleaned.loc[1,"cp"])
    filled, modes = impute_mode(cleaned, cols=["cp"])
    filled, medians = impute_median(filled, cols=["trestbps"])
    assert filled["cp"].tolist() == [1,1] and modes["cp"] == 1
    assert filled["trestbps"].notna().all() and medians["trestbps"] == 120.0
    X, y = split_features_target(filled)
    assert TARGET_COL not in X.columns and y.equals(filled[TARGET_COL])
    report = data_quality_report(filled)
    assert report["n_rows"] == 2 and report["n_missing"] == 0
    return f"dedup={len(dedup)}, mode={modes}, median={medians}, target separated"

CONFIRMED_VALID_CODES = {"cp": {5}, "slope": {4}, "ca": {4}}

def test_confirmed_valid_codes_in_expected_values():
    missing = {
        col: sorted(values - EXPECTED_VALUES.get(col, set()))
        for col, values in CONFIRMED_VALID_CODES.items()
        if values - EXPECTED_VALUES.get(col, set())
    }
    if missing:
        raise AssertionError(
            f"User confirmed these as valid, but src.data.EXPECTED_VALUES excludes them: {missing}. "
            "replace_abnormal_with_nan therefore removes these codes before imputation; update the domain "
            "and regenerate processed data/metrics before using results as final."
        )
    return "All user-confirmed valid codes are included in EXPECTED_VALUES"

DATA_TESTS = [
    ("loader all-source/filter", test_loader_reads_all_sources_and_filters_source),
    ("loader error paths", test_loader_errors),
    ("dedup/abnormal/imputation/split", test_duplicate_abnormal_imputation_and_split),
    ("user-confirmed valid codes included in EXPECTED_VALUES", test_confirmed_valid_codes_in_expected_values),
]
for test_name,test_fn in DATA_TESTS:
    test_case("DATA / " + test_name,test_fn,note="User confirmed cp=5, slope=4, ca=4 valid; source data is not being labeled invalid." if "user-confirmed" in test_name else "")


## 4. Preprocessing, metric và adapter thư viện
Dùng fixture nhỏ để kiểm tra số cột Manhattan, tính chất ma trận Gower, metric đã biết và adapter thư viện cho feature/precomputed. Không chạy sweep.

In [ ]:
def test_manhattan_preprocessing_schema():
    frame = pd.DataFrame({
        "age":[40,50,60,70],"trestbps":[120,130,140,150],"chol":[200,210,220,230],"thalach":[160,150,140,130],"oldpeak":[0,.5,1,1.5],
        "cp":[1,2,3,4],"restecg":[0,1,2,0],"slope":[1,2,3,1],"ca":[0,1,2,3],"thal":[3,6,7,3],
        "sex":[0,1,0,1],"fbs":[0,0,1,1],"exang":[0,1,0,1]})
    matrix, names, transformer = preprocess_features(frame)
    assert matrix.shape == (4,25) and len(names)==25 and np.isfinite(matrix).all()
    assert "cp_1" in names and "cp_4" in names
    return f"fixture produces {matrix.shape[1]} finite Manhattan features"

def test_preprocessing_missing_columns_and_gower_properties():
    expect_raises(ValueError,lambda:preprocess_features(pd.DataFrame({"age":[1]})))
    frame=pd.DataFrame({
        "age":[40,50,60],"trestbps":[120,130,140],"chol":[200,210,220],"thalach":[160,150,140],"oldpeak":[0,.5,1],
        "cp":[1,2,4],"restecg":[0,1,2],"slope":[1,2,3],"ca":[0,1,3],"thal":[3,6,7],"sex":[0,1,0],"fbs":[0,1,0],"exang":[0,1,0]})
    D=gower_distance_matrix(frame)
    assert D.shape==(3,3) and np.isfinite(D).all()
    assert np.allclose(D,D.T) and np.allclose(np.diag(D),0)
    assert (D>=0).all() and (D<=1).all()
    expect_raises(ValueError,lambda:gower_distance_matrix(frame.assign(age=[40,np.nan,60])))
    return f"Gower symmetric, diagonal=0, range=[{D.min():.3f},{D.max():.3f}]"

def test_metrics_known_fixtures():
    labels=np.array([0,0,1,1]); y=pd.Series([0,0,1,1],name=TARGET_COL)
    ext=posthoc_target_check(labels,y)
    assert np.isclose(ext["ARI"],1) and np.isclose(ext["purity"],1)
    assert cluster_sizes(labels).to_dict()=={0:2,1:2}
    assert compare_labelings(labels,labels)==1.0
    assert np.isnan(silhouette(np.zeros((4,1)),np.zeros(4),metric="euclidean"))
    D=np.array([[0,1,9,10],[1,0,8,9],[9,8,0,1],[10,9,1,0]],float)
    score=silhouette(D,labels,metric="precomputed")
    assert 0 < score <= 1
    return f"known ARI/purity=1; precomputed silhouette={score:.3f}"

def test_profile_and_target_crosstab_fixture():
    frame=pd.DataFrame({"age":[20,30,50,60],"trestbps":[100,110,140,150],"chol":[180,190,230,240],"thalach":[180,170,140,130],"oldpeak":[0,.1,2,3],"sex":[0,1,1,1],"fbs":[0,0,1,1],"exang":[0,0,1,1],"target":[0,0,1,1]})
    labels=np.array([0,0,1,1])
    profile=cluster_profile(frame,labels)
    assert profile["n"].tolist()==[2,2] and profile["target_rate"].tolist()==[0.0,1.0]
    ct=target_crosstab(labels,frame[TARGET_COL])
    assert np.allclose(ct["tỉ_lệ_bệnh"].to_numpy(),[0,1])
    return "cluster sizes and post-hoc rates match fixture"

def test_library_adapter_modes():
    X=np.array([[0.,0.],[0.,1.],[10.,10.],[10.,11.]])
    D=np.abs(X[:,None,:]-X[None,:,:]).sum(axis=2)
    a=fit_kmedoids(X,2,metric="manhattan",random_state=42)
    b=fit_kmedoids(D,2,metric="precomputed",random_state=42)
    assert a.labels.shape==(4,) and a.medoid_indices.shape==(2,) and np.isfinite(a.inertia)
    assert b.labels.shape==(4,) and b.medoid_indices.shape==(2,) and np.isfinite(b.inertia)
    expect_raises(ValueError,lambda:fit_kmedoids(np.zeros((3,2)),2,metric="precomputed"))
    return f"feature/precomputed adapter outputs have {len(a.labels)} labels"

def test_select_k_pipeline_fixture():
    X=np.array([[0.,0.],[0.,1.],[1.,0.],[9.,9.],[10.,9.],[9.,10.]])
    best,table,results=select_k(X,metric="manhattan",k_range=[2,3],random_state=42)
    assert best==2 and set(table.index)=={2,3} and set(results)=={2,3}
    assert {"silhouette","inertia","min_cluster_size","max_cluster_size"}.issubset(table.columns)
    assert all(np.isfinite(table[c]).all() for c in ["silhouette","inertia"])
    assert all(len(r.labels)==len(X) and len(r.medoid_indices)==k for k,r in results.items())
    assert table.loc[2,"silhouette"]>table.loc[3,"silhouette"]
    return f"select_k chose k={best}; scores={table.silhouette.round(3).to_dict()}"

def test_processed_artifact_roundtrip_in_temporary_directory():
    frame=pd.DataFrame({
        "age":[30,35,40,45,50,55],"trestbps":[110,115,120,125,130,135],"chol":[180,185,190,195,200,205],"thalach":[180,175,170,165,160,155],"oldpeak":[0,.1,.2,.3,.4,.5],
        "cp":[1,2,3,4,1,2],"restecg":[0,1,2,0,1,2],"slope":[1,2,3,1,2,3],"ca":[0,1,2,3,0,1],"thal":[3,6,7,3,6,7],
        "sex":[0,1,0,1,0,1],"fbs":[0,0,1,1,0,1],"exang":[0,1,0,1,0,1],"target":[0,1,0,1,0,1]})
    X_mat,names,_=preprocess_features(frame[FEATURE_COLS])
    D=gower_distance_matrix(frame[FEATURE_COLS])
    with TemporaryDirectory(prefix="heart_kmedoids_test_processed_") as temp:
        paths=save_processed(frame,X_mat,names,D,Path(temp)/"processed")
        base=pd.read_csv(paths["dedup"]); X_saved=pd.read_csv(paths["X"]); y=pd.read_csv(paths["target"])
        D_saved=np.load(paths["gower"])
        assert base.equals(frame) and X_saved.columns.tolist()==names
        assert np.allclose(X_saved.to_numpy(),X_mat) and y[TARGET_COL].equals(frame[TARGET_COL])
        assert np.allclose(D_saved,D)
    return "save_processed write/read roundtrip matches fixture; temporary files cleaned"

def test_metrics_pipeline_writes_and_reads_temp_outputs():
    # Nine distinct synthetic rows allow the project's fixed k=2..8 sweep.
    frame=pd.DataFrame({
        "age":[31,34,38,42,47,53,58,64,70],"trestbps":[108,116,124,132,140,148,156,164,172],"chol":[175,186,197,208,219,230,241,252,263],"thalach":[190,182,174,166,158,150,142,134,126],"oldpeak":[0,.1,.2,.3,.4,.5,.6,.7,.8],
        "cp":[1,2,3,4,1,2,3,4,1],"restecg":[0,1,2,0,1,2,0,1,2],"slope":[1,2,3,1,2,3,1,2,3],"ca":[0,1,2,3,0,1,2,3,0],"thal":[3,6,7,3,6,7,3,6,7],
        "sex":[0,1,0,1,0,1,0,1,0],"fbs":[0,0,1,1,0,0,1,1,0],"exang":[0,1,0,1,0,1,0,1,0],"target":[0,1,0,1,0,1,0,1,0]})
    with TemporaryDirectory(prefix="heart_kmedoids_test_metrics_") as temp:
        paths=run_and_save_metrics(frame,temp,extra_config={"test_only":True})
        expected={"k_selection","labels","medoids","cross_method_ari","run_config"}
        assert set(paths)==expected and all(Path(path).is_file() for path in paths.values())
        metrics=pd.read_csv(paths["k_selection"]); labels=pd.read_csv(paths["labels"])
        medoids=pd.read_csv(paths["medoids"]); cross=pd.read_csv(paths["cross_method_ari"])
        config=json.loads(Path(paths["run_config"]).read_text(encoding="utf-8"))
        assert len(metrics)==14 and set(metrics["k"])==set(range(2,9))
        assert labels.shape==(len(frame),14) and len(medoids)==2*sum(range(2,9))
        assert cross["k"].tolist()==list(range(2,9))
        assert config["n_rows"]==len(frame) and config["test_only"] is True
    return "run_and_save_metrics outputs/files/row counts/config roundtrip passed; temporary outputs cleaned"

PREP_EVAL_TESTS=[
    ("Manhattan feature schema",test_manhattan_preprocessing_schema),
    ("Gower matrix validation",test_preprocessing_missing_columns_and_gower_properties),
    ("metrics known fixtures",test_metrics_known_fixtures),
    ("cluster profile/crosstab",test_profile_and_target_crosstab_fixture),
    ("library feature/precomputed adapter",test_library_adapter_modes),
    ("select_k pipeline",test_select_k_pipeline_fixture),
    ("processed artifact roundtrip (temporary directory)",test_processed_artifact_roundtrip_in_temporary_directory),
    ("metrics save/read integration (temporary directory)",test_metrics_pipeline_writes_and_reads_temp_outputs),
]
for test_name,test_fn in PREP_EVAL_TESTS:
    test_case("PREPROCESS/EVALUATION/LIBRARY / "+test_name,test_fn)

## 5. Artifact tích hợp và full-data smoke test read-only
Đọc cấu hình/labels/medoids hiện có, kiểm tra căn hàng. Fit custom với k=2 trên X Manhattan trong bộ nhớ, tính silhouette/ARI/purity và ARI so với nhãn thư viện đã lưu; không ghi kết quả ra file.

In [ ]:
def test_saved_artifacts_and_config_are_aligned():
    raw=pd.read_csv(RAW_PATH); base=pd.read_csv(PROCESSED_DIR/"heart_dedup.csv")
    X=pd.read_csv(PROCESSED_DIR/"X_manhattan.csv"); y=pd.read_csv(PROCESSED_DIR/"target.csv")[TARGET_COL]
    D=np.load(PROCESSED_DIR/"gower_distance.npy",mmap_mode="r")
    labels=pd.read_csv(METRICS_DIR/"labels.csv"); medoids=pd.read_csv(METRICS_DIR/"medoids.csv")
    metrics=pd.read_csv(METRICS_DIR/"k_selection.csv"); config=json.loads((METRICS_DIR/"run_config.json").read_text(encoding="utf-8"))
    assert len(raw)==len(base)==len(X)==len(y)==len(labels)==2283
    assert D.shape==(2283,2283) and base[TARGET_COL].equals(y) and TARGET_COL not in X.columns
    assert list(base.columns)==FEATURE_COLS+[TARGET_COL]
    assert set(metrics["distance"])=={"manhattan","gower"} and set(metrics["k"])==set(range(2,9))
    assert set(labels.columns)=={f"{d}_k{k}" for d in ["manhattan","gower"] for k in range(2,9)}
    assert medoids["medoid_index"].between(0,2282).all()
    assert config["n_rows"]==2283 and config["n_features_manhattan"]==25 and config["source_filter"] is None
    return "2,283-row alignment, target separation, labels/medoids/config schemas pass"

def test_preprocessing_reproduces_saved_processed_artifacts():
    raw=load_heart_data(RAW_PATH)
    cleaned=replace_abnormal_with_nan(raw)
    # Match Notebook 1: drop exact duplicates before filling missing values.
    cleaned=drop_duplicate_rows(cleaned)
    cleaned,fill_cat=impute_mode(cleaned,cols=CATEGORICAL_COLS+BINARY_COLS)
    cleaned,fill_num=impute_median(cleaned,cols=NUMERIC_COLS)
    saved=pd.read_csv(PROCESSED_DIR/"heart_dedup.csv")
    assert len(cleaned)==len(saved)==2283
    pd.testing.assert_frame_equal(cleaned[FEATURE_COLS+[TARGET_COL]],saved[FEATURE_COLS+[TARGET_COL]],check_dtype=False,check_exact=False,rtol=0,atol=0)
    X_now=pd.read_csv(PROCESSED_DIR/"X_manhattan.csv")
    X_rebuilt,names,_=preprocess_features(cleaned[FEATURE_COLS])
    assert names==X_now.columns.tolist()
    assert np.allclose(X_rebuilt,X_now.to_numpy(dtype=float),rtol=0,atol=1e-12)
    D_now=np.load(PROCESSED_DIR/"gower_distance.npy",mmap_mode="r")
    D_rebuilt=gower_distance_matrix(cleaned[FEATURE_COLS])
    assert np.allclose(D_rebuilt,D_now,rtol=0,atol=1e-12)
    return f"rebuild matches processed/Gower artifacts; impute mode={fill_cat}; median={fill_num}"

def test_read_only_full_data_custom_smoke():
    base=pd.read_csv(PROCESSED_DIR/"heart_dedup.csv")
    X=pd.read_csv(PROCESSED_DIR/"X_manhattan.csv")
    y=pd.read_csv(PROCESSED_DIR/"target.csv")[TARGET_COL]
    assert len(X)==len(base)==len(y)==2283 and TARGET_COL not in X.columns
    m=KMedoids(2,metric="manhattan",random_state=42,max_iter=100).fit(X.to_numpy(dtype=float))
    assert len(m.labels_)==2283 and sum(m.cluster_sizes_)==2283
    assert m.converged_ and m.medoid_indices_==[2018,88]
    assert m.cluster_sizes_==[2084,199] and np.isclose(m.inertia_,17072.37170522124)
    external=posthoc_target_check(m.labels_,y)
    assert 0<=external["purity"]<=1 and -1<=external["ARI"]<=1
    print({"k":2,"metric":"Manhattan","seed":42,"medoid_index_0based":m.medoid_indices_,"cluster_sizes":m.cluster_sizes_,"objective":m.inertia_,"n_iter":m.n_iter_,"converged":m.converged_,"posthoc":external})
    return "custom full-data run passed in memory; no artifact saved"

def test_custom_vs_saved_library_label_ari_read_only():
    X=pd.read_csv(PROCESSED_DIR/"X_manhattan.csv").to_numpy(dtype=float)
    saved=pd.read_csv(METRICS_DIR/"labels.csv")["manhattan_k2"].to_numpy()
    custom=KMedoids(2,metric="manhattan",random_state=42,max_iter=100).fit(X)
    assert len(saved)==len(custom.labels_)==2283
    score=compare_labelings(np.asarray(custom.labels_),saved)
    assert np.isfinite(score) and -1<=score<=1
    print(f"ARI custom vs saved library Manhattan k=2: {score:.6f}")
    return f"ARI={score:.6f}; comparison is read-only"

ARTIFACT_TESTS=[
    ("saved artifact/config alignment",test_saved_artifacts_and_config_are_aligned),
    ("rebuild preprocessing vs saved artifacts",test_preprocessing_reproduces_saved_processed_artifacts),
    ("custom full-data k=2 smoke (in memory)",test_read_only_full_data_custom_smoke),
    ("custom-vs-saved-library labels ARI",test_custom_vs_saved_library_label_ari_read_only),
]
for test_name,test_fn in ARTIFACT_TESTS:
    test_case("ARTIFACTS/INTEGRATION / "+test_name,test_fn)

def test_no_protected_artifact_was_modified():
    changed=[]
    for path,digest in FILE_HASHES_BEFORE.items():
        if not path.exists() or sha256(path.read_bytes()).hexdigest()!=digest:
            changed.append(str(path.relative_to(PROJECT_ROOT)))
    assert not changed,f"protected data/metric artifacts changed: {changed}"
    return f"{len(FILE_HASHES_BEFORE)} raw/processed/metric files unchanged"

test_case("READ-ONLY SAFETY / protected artifacts unchanged",test_no_protected_artifact_was_modified)


## 6. Tổng hợp PASS/FAIL
Nếu miền `EXPECTED_VALUES` chưa gồm các mã đã được người dùng xác nhận, bài test code-contract sẽ FAIL bắt buộc; đây là lỗi cần sửa trong pipeline, không phải kết luận dữ liệu không hợp lệ. Các test khác vẫn chạy để tổng hợp lỗi; artifact dữ liệu/metrics được bảo vệ bằng hash.

In [ ]:
results_df = pd.DataFrame(TEST_RESULTS)
display(results_df)
required_failures = results_df[(results_df.status=="FAIL") & results_df.required]
optional_failures = results_df[(results_df.status=="FAIL") & (~results_df.required)]
print(f"Required checks: {len(results_df)-len(required_failures)-len(optional_failures)}/{len(results_df)} passed; required failures={len(required_failures)}; advisory/data-contract failures={len(optional_failures)}")
if len(optional_failures):
    print("Advisory failures need owner review; they are not silently suppressed or auto-fixed.")
    display(optional_failures[["test","detail","note"]])
if len(required_failures):
    display(required_failures[["test","detail","note"]])
    raise AssertionError(f"{len(required_failures)} required notebook checks failed.")
print("All required Notebook 03 regression/integration checks passed.")
